# Harvest readiness, run 3: leave one farm out

**Why run 3.** Run 2 scored 0.87 to 0.90 on the same farm but 0.00 on the other farm, in both directions. Two things explain it:

* **Camera distance.** Duman photos are wide greenhouse shots with tiny clusters; angelo photos are close-ups.
* **Leaked near-twins.** Duman is a time series (file names are millisecond timestamps: 555 photos over 75 days, two cycles, often 2 to 3 shots within minutes). A random split put near-identical photos on both sides, so the same-farm score was too optimistic.

**What changed.** This notebook is edited in place from run 2 (git keeps the run 2 version).

1. Five farms instead of two: Duman plus four Roboflow sets (angelo, amr, thesis, ros2). A set that cannot be downloaded is skipped and named.
2. Photos that belong together stay together: Duman photos from the same calendar day form one group; for the other farms, near-identical photos (perceptual hash) form one group. Copies across farms are removed.
3. Each farm is capped at 600 training photos (whole groups).
4. **Leave one farm out:** train on the other farms (85/15 train/check split by group, never random by photo) and score on ALL photos of the left-out farm.
5. The score table has one extra column, **clusters found (ignoring young/ready)**: the same predictions scored as if every cluster had one label. If it is high but young/ready are low, the model finds clusters but mislabels them; if it is low too, it fails at finding them.

| Farm | Source | Their labels | Our label |
|---|---|---|---|
| duman | Mendeley hf55tkx489 (Duman et al. 2024), uploaded to Kaggle | Immature / Mature | young / ready |
| angelo | Roboflow angelo-neyra-9rg4y / mushroom-w7ucu | NOT_READY / READY, OVERDUE | young / ready |
| amr | Roboflow amr-asmr / oyster-mushroom-growth | Juvenile, Intermediate / Harvest | young / ready |
| thesis | Roboflow thesis-fgp1n / oyster-mushroom-v2 | Not ready / Ready | young / ready |
| ros2 | Roboflow ros2-plan / oyster-mushroom-bkezg | immature / mature | young / ready |

**Before you press Run All (once):**
1. Duman must be a private Kaggle dataset (see the box below).
2. Right panel, *Session options*: Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
3. *Add-ons, Secrets*: switch `ROBOFLOW_API_KEY` on for this notebook. The key is never printed or saved.
4. *Add Input*: add your Duman dataset.

Runtime: five trainings of 40 epochs at 960 px, roughly 2 to 4 hours in total. At the end one table sums up everything; a screenshot is enough. syldd photos are never used here.

### Uploading Duman to Kaggle (one time)
On kaggle.com: **Create, New Dataset**, title `duman-oyster-maturity`, keep it **Private**. Drag in these two folders from `mycodemo/data/external/d010_duman2024/`:

* `Raw Images` (555 photos, 270 MB)
* `Maturity Labelled Images and Annotation Files/YOLO Format` (one small .txt label file per photo)

Press **Create**. The notebook finds the two folders by what is inside them (not by their names), so renamed or flattened folders are fine.

In [ ]:
# 1. Install the libraries we need (Kaggle has the rest)
!pip install -q roboflow ultralytics imagehash

In [ ]:
# 2. Settings: change these lines only if you want to
import re
def norm(s):
    # match label names ignoring case, spaces, dashes and underscores
    return re.sub(r"[\s_\-]+", "", str(s).lower())

# farm: (Roboflow workspace, project, {their label: our label})
ROBOFLOW = {
    "angelo": ("angelo-neyra-9rg4y", "mushroom-w7ucu",
               {"not_ready": "young", "ready": "ready", "overdue": "ready"}),
    "amr":    ("amr-asmr", "oyster-mushroom-growth",
               {"juvenile": "young", "intermediate": "young", "harvest": "ready"}),
    "thesis": ("thesis-fgp1n", "oyster-mushroom-v2",
               {"not ready": "young", "ready": "ready"}),
    "ros2":   ("ros2-plan", "oyster-mushroom-bkezg",
               {"immature": "young", "mature": "ready"}),
}
DUMAN_LABELS = {"immature": "young", "mature": "ready"}
MAPPING = {"duman": DUMAN_LABELS, **{k: v[2] for k, v in ROBOFLOW.items()}}
MAPPING = {f: {norm(k): v for k, v in m.items()} for f, m in MAPPING.items()}

OUR_LABELS = ["young", "ready"]   # class 0 = young, class 1 = ready
EPOCHS = 40                       # passes over the training photos
IMG_SIZE = 960                    # Duman clusters are small in a wide photo
MODEL = "yolo11n.pt"              # smallest rectangle model
CAP = 600                         # max training photos per farm (whole groups)
CHECK_FRAC = 0.15                 # share of training groups held back to watch progress
HASH_DIST = 4                     # perceptual-hash distance (of 64 bits) counted as "near-identical"
SEED = 0
WORK = "/kaggle/working"

In [ ]:
# 3. Find the Duman folders you uploaded (by content, not by folder name)
import os, glob, collections
PHOTO_EXT = (".jpg", ".jpeg")

def list_images(d):
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(PHOTO_EXT))

def show_tree():
    for root, dirs, files in sorted(os.walk("/kaggle/input")):
        kinds = collections.Counter(os.path.splitext(f)[1].lower() or "(no extension)" for f in files)
        depth = root.count(os.sep) - 2
        print("  " * depth + os.path.basename(root) + "/", dict(kinds) if kinds else "")

def find_dir():
    best_photos, best_labels = (0, None), (0, None)
    for root, dirs, files in os.walk("/kaggle/input"):
        n_img = sum(f.lower().endswith(PHOTO_EXT) for f in files)
        if n_img > best_photos[0]:
            best_photos = (n_img, root)
        if "classes.txt" in files:
            names = [n.lower() for n in open(os.path.join(root, "classes.txt"), errors="ignore").read().split()]
            n_txt = sum(f.lower().endswith(".txt") for f in files)
            # the Duman maturity labels list Mature and Immature; the other YOLO folder does not
            if "mature" in names and "immature" in names and n_txt > best_labels[0]:
                best_labels = (n_txt, root)
    if best_photos[1] is None or best_labels[0] < 10:
        print("Could not find the Duman photos and labels. What is under /kaggle/input:")
        show_tree()
        raise FileNotFoundError("Duman photos and/or labels not found under /kaggle/input. "
                                "Attach the Duman dataset to this notebook with Add Input (right panel), then run again.")
    return best_photos[1], best_labels[1]

DUMAN_IMG, DUMAN_LAB = find_dir()
print("photos:", DUMAN_IMG, len(list_images(DUMAN_IMG)))
print("labels:", DUMAN_LAB, len(glob.glob(DUMAN_LAB + "/*.txt")))
print("their label names:", open(DUMAN_LAB + "/classes.txt").read().split())

In [ ]:
# 4. Download the four Roboflow sets (version with the fewest photos = no flipped/augmented copies)
#    A set that fails is skipped and named; the key is never printed.
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=KEY)
DOWNLOADS, SKIPPED = {}, {}
for short, (ws, slug, _) in ROBOFLOW.items():
    try:
        proj = rf.workspace(ws).project(slug)
        vers = []
        for v in proj.versions():
            n = getattr(v, "images", None)
            if not n:
                splits = getattr(v, "splits", None) or {}
                n = sum(int(x) for x in splits.values()) if splits else 10**9
            vers.append((int(n), int(str(v.version).split("/")[-1])))
        ver = sorted(vers)[0][1]
        DOWNLOADS[short] = proj.version(ver).download("yolov11", location=f"{WORK}/raw_{short}", overwrite=True).location
        print(f"{short}: versions (photos, version) {sorted(vers)} -> chose version {ver}")
    except Exception as e:
        SKIPPED[short] = f"{type(e).__name__}: {str(e).replace(KEY, '***')[:200]}"
        print(f"SKIPPED {short} ({ws}/{slug}): {SKIPPED[short]}")
print("\nfarms downloaded:", list(DOWNLOADS), "| skipped:", list(SKIPPED))

## Put all farms into the same shape
Every label is rewritten as 0 = young, 1 = ready (names matched ignoring case, spaces, dashes, underscores). Traced outlines become the smallest rectangle around them. Labels not in the tables above are dropped and counted per farm, so you can see what was thrown away.

In [ ]:
import shutil, random, yaml, collections, glob, datetime
import numpy as np, pandas as pd
from PIL import Image

def to_box(parts):
    # 'class x y w h' stays; 'class x1 y1 x2 y2 ...' (outline) becomes its surrounding rectangle
    nums = [float(p) for p in parts[1:]]
    if len(nums) == 4:
        return nums
    xs, ys = nums[0::2], nums[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0]

ORIG = {}   # pooled photo path -> original file name (Duman: the timestamp)

def pool(short, pairs, their, mapping):
    # pairs = list of (image path, label path); their = list of their label names
    new_id = {i: OUR_LABELS.index(mapping[norm(n)]) for i, n in enumerate(their) if norm(n) in mapping}
    out = f"{WORK}/pool_{short}"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(f"{out}/images"); os.makedirs(f"{out}/labels")
    counts, dropped = collections.Counter(), collections.Counter()
    for k, (img, lab) in enumerate(pairs):
        lines = []
        if lab and os.path.exists(lab):
            for line in open(lab):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                if c in new_id:
                    lines.append(" ".join([str(new_id[c])] + [f"{v:.6f}" for v in to_box(parts)]))
                    counts[OUR_LABELS[new_id[c]]] += 1
                else:
                    dropped[their[c] if c < len(their) else f"unknown class {c}"] += 1
        name = f"{short}_{k:05d}"
        dst = f"{out}/images/{name}{os.path.splitext(img)[1]}"
        shutil.copy(img, dst)
        ORIG[dst] = os.path.basename(img)
        open(f"{out}/labels/{name}.txt", "w").write("\n".join(lines))
    print(f"{short}: {len(pairs)} photos, clusters {dict(counts)}, DROPPED labels {dict(dropped)}, "
          f"{sum(counts.values()) / max(len(pairs), 1):.1f} clusters per photo")
    return out

POOL = {}
# duman: one label file per raw photo, same file name
dum_names = open(DUMAN_LAB + "/classes.txt").read().split()
POOL["duman"] = pool("duman", [(img, f"{DUMAN_LAB}/{os.path.splitext(os.path.basename(img))[0]}.txt")
                               for img in list_images(DUMAN_IMG)], dum_names, MAPPING["duman"])

# Roboflow farms: pool train/valid/test together (the held-out farm is scored on everything anyway)
for short, dl in DOWNLOADS.items():
    names = yaml.safe_load(open(f"{dl}/data.yaml"))["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    pairs = []
    for split in ["train", "valid", "test"]:
        for img in sorted(glob.glob(f"{dl}/{split}/images/*")):
            base = os.path.splitext(os.path.basename(img))[0]
            pairs.append((img, f"{dl}/{split}/labels/{base}.txt"))
    POOL[short] = pool(short, pairs, names, MAPPING[short])

## Groups and duplicates
* **Duman:** photos from the same calendar day (UTC, from the millisecond time stamp in the file name) form one group.
* **Other farms:** photos whose perceptual hashes differ by at most `HASH_DIST` bits form one group.
* **Across farms:** a photo that is near-identical to a photo of an earlier farm (Duman first) is removed.

Splits always move whole groups.

In [ ]:
import imagehash

def phash_int(path):
    with Image.open(path) as im:
        h = imagehash.phash(im.convert("RGB")).hash.flatten()
    return int(np.packbits(h).view(">u8")[0])

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def dist_to_all(H, h):
    # Hamming distance from hash h to every hash in H
    return POP[(H ^ np.uint64(h)).view(np.uint8)].reshape(-1, 8).sum(axis=1)

PHOTOS = {}      # farm -> list of (photo path, group id)
kept_hashes = np.zeros(0, dtype=np.uint64)
removed = {}
for short in POOL:                      # duman is first, so its photos win the cross-farm duplicates
    imgs = sorted(glob.glob(f"{POOL[short]}/images/*"))
    H = np.array([phash_int(p) for p in imgs], dtype=np.uint64)
    keep = np.ones(len(imgs), bool)
    if len(kept_hashes):
        for i in range(len(imgs)):
            if dist_to_all(kept_hashes, int(H[i])).min() <= HASH_DIST:
                keep[i] = False
    removed[short] = int((~keep).sum())
    for p in np.array(imgs)[~keep]:
        os.remove(p); os.remove(p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt")
    imgs, H = list(np.array(imgs)[keep]), H[keep]
    if short == "duman":
        def day(p):
            try:
                return str(datetime.datetime.fromtimestamp(int(os.path.splitext(ORIG[p])[0]) / 1000, datetime.timezone.utc).date())
            except ValueError:
                return None
        days = [day(p) for p in imgs]
        if None in days:
            print("WARNING: some Duman file names are not timestamps; those photos form their own groups")
        gid = [f"duman_{d}" if d else f"duman_single_{i}" for i, d in enumerate(days)]
    else:                                # union-find over near-identical hashes
        parent = list(range(len(imgs)))
        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]; a = parent[a]
            return a
        for i in range(len(imgs)):
            for j in np.nonzero(dist_to_all(H[:i], int(H[i])) <= HASH_DIST)[0] if i else []:
                parent[find(i)] = find(int(j))
        gid = [f"{short}_g{find(i)}" for i in range(len(imgs))]
    PHOTOS[short] = list(zip(imgs, gid))
    kept_hashes = np.concatenate([kept_hashes, H])

print("farm     photos  groups  (removed as copies of an earlier farm)")
for s, items in PHOTOS.items():
    print(f"{s:8s} {len(items):6d} {len(set(g for _, g in items)):7d}  ({removed[s]})")

## Data check: what do the photos and labels look like?
Different photo sizes between farms, or tiny clusters, can explain a bad score on their own. A strong imbalance between young and ready can bias the model towards the more common label.

In [ ]:
# Data check on the pooled, de-duplicated sets (labels are already 0 = young, 1 = ready)
def describe(short):
    imgs = [p for p, _ in PHOTOS[short]]
    sizes, modes, per_photo, shares = collections.Counter(), collections.Counter(), [], []
    labels = collections.Counter()
    for img in imgs:
        with Image.open(img) as im:
            sizes[f"{im.size[0]}x{im.size[1]}"] += 1
            modes[im.mode] += 1
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        rows = [l.split() for l in open(lab).read().splitlines() if l.strip()]
        per_photo.append(len(rows))
        for c, x, y, w, h in rows:
            labels[OUR_LABELS[int(c)]] += 1
            shares.append(float(w) * float(h) * 100)
    total = max(sum(labels.values()), 1)
    per_photo, shares = np.array(per_photo), np.array(shares)
    fmt = lambda cnt: ", ".join(f"{k}: {v}" for k, v in cnt.most_common(4))
    rows = [("photos", len(imgs)),
            ("photo sizes (width x height)", fmt(sizes)),
            ("colour channels (PIL mode)", fmt(modes))]
    rows += [(f"clusters, {lab}", f"{labels[lab]} ({100 * labels[lab] / total:.0f}%)") for lab in OUR_LABELS]
    rows += [("photos with no clusters", int((per_photo == 0).sum())),
             ("clusters per photo (median, max)", f"{np.median(per_photo):.0f}, {per_photo.max()}"),
             ("cluster size, % of photo (median, 10th percentile)",
              f"{np.median(shares):.1f}%, {np.percentile(shares, 10):.1f}%" if len(shares) else "n/a")]
    return pd.DataFrame(rows, columns=["check", short]).set_index("check")

print(pd.concat([describe(s) for s in PHOTOS], axis=1).to_string())

## Label check: look before you trust
Six random photos from each farm with the **dataset's own labels** drawn on (not the model's). Blue = young, orange = ready. Check by eye:
* Does each rectangle hold one whole cluster, or single caps?
* Do the farms agree on what counts as ready?

If the labels look wrong here, no score below can be trusted.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
COLOURS = {0: "tab:blue", 1: "tab:orange"}

def show_labels(short, n=6, seed=1):
    imgs = [p for p, _ in PHOTOS[short]]
    random.seed(seed)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, img in zip(axes.flat, random.sample(imgs, min(n, len(imgs)))):
        im = Image.open(img); W, H = im.size
        ax.imshow(im); ax.axis("off")
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        for line in open(lab).read().splitlines():
            c, x, y, w, h = line.split(); c = int(c)
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False,
                                           edgecolor=COLOURS[c], linewidth=2))
    fig.suptitle(f"{short}: dataset's own labels (blue = young, orange = ready)", fontsize=14)
    plt.tight_layout(); plt.show()

for s in PHOTOS:
    show_labels(s)

In [ ]:
# 5. Settings files, one run per left-out farm.
#    Training photos: other farms, capped at CAP per farm (whole groups, seed 0), then 85/15 split by GROUP.
#    Test photos: ALL photos of the left-out farm, never seen in training.
def cap_groups(items):
    groups = collections.defaultdict(list)
    for p, g in items:
        groups[g].append(p)
    order = sorted(groups); random.Random(SEED).shuffle(order)
    kept, n = [], 0
    for g in order:
        if n + len(groups[g]) <= CAP:
            kept.append(g); n += len(groups[g])
    return {g: groups[g] for g in kept}

def split_by_group(groups):
    order = sorted(groups); random.Random(SEED).shuffle(order)
    total = sum(len(v) for v in groups.values())
    check, n = [], 0
    for g in order:
        if n < CHECK_FRAC * total:
            check.append(g); n += len(groups[g])
    train = [p for g in order if g not in check for p in groups[g]]
    return train, [p for g in check for p in groups[g]]

CAPPED = {s: cap_groups(items) for s, items in PHOTOS.items()}
print("training photos per farm after the cap:", {s: sum(len(v) for v in g.values()) for s, g in CAPPED.items()})

def make_run(left_out):
    train, val = [], []
    for s in PHOTOS:
        if s != left_out:
            t, v = split_by_group(CAPPED[s]); train += t; val += v
    run = f"{WORK}/run_leave_out_{left_out}"
    os.makedirs(run, exist_ok=True)
    open(f"{run}/train.txt", "w").write("\n".join(sorted(train)))
    open(f"{run}/val.txt", "w").write("\n".join(sorted(val)))
    open(f"{run}/test.txt", "w").write("\n".join(sorted(p for p, _ in PHOTOS[left_out])))
    cfg = {"path": run, "train": f"{run}/train.txt", "val": f"{run}/val.txt",
           "test": f"{run}/test.txt", "names": {0: "young", 1: "ready"}}
    yaml.safe_dump(cfg, open(f"{run}/data.yaml", "w"))
    print(f"leave out {left_out}: train {len(train)}, check {len(val)}, test {len(PHOTOS[left_out])}")
    return run

RUNS = {s: make_run(s) for s in PHOTOS if len(PHOTOS) > 1}

## Train and score
The score is **mAP50 (box)**: roughly, the share of clusters the model puts a rectangle on in the right place *and* with the right label, from 0 (useless) to 1 (perfect). Rough guide: above 0.5 on the unseen farm is a usable first model; below 0.3 means the farms still look too different.

* **young / ready / all:** the usual score with labels.
* **clusters found (ignoring young/ready):** the same predictions scored with every cluster set to one class. High here but low above = the model finds clusters but mislabels them. Low here too = it fails at finding them.
* **check (other farms):** the 15% held-back groups of the training farms. Because whole groups are held back, this is no longer inflated by near-twin photos.

In [ ]:
from ultralytics import YOLO
results = []
def class_scores(r):
    out = {}
    for i, lab in enumerate(OUR_LABELS):
        try:
            out[lab] = round(r.box.class_result(i)[2], 3)
        except Exception:
            out[lab] = None      # that label does not occur in the photos scored
    return out

for left_out, run in RUNS.items():
    print("\n##### leave out:", left_out)
    model = YOLO(MODEL)
    model.train(data=f"{run}/data.yaml", epochs=EPOCHS, imgsz=IMG_SIZE, seed=SEED,
                project=run, name="train", exist_ok=True, verbose=False, plots=True)
    best = YOLO(f"{run}/train/weights/best.pt")
    own = best.val(data=f"{run}/data.yaml", split="val", imgsz=IMG_SIZE, project=run, name="val_check", exist_ok=True)
    other = best.val(data=f"{run}/data.yaml", split="test", imgsz=IMG_SIZE, project=run, name="test_left_out", exist_ok=True)
    # single_cls=True scores the same predictions with every class merged into one
    one = best.val(data=f"{run}/data.yaml", split="test", imgsz=IMG_SIZE, single_cls=True,
                   project=run, name="test_left_out_one_class", exist_ok=True)
    sc = class_scores(other)
    results.append({"left-out farm": left_out,
                    "test photos": len(PHOTOS[left_out]),
                    "mAP50 young": sc["young"], "mAP50 ready": sc["ready"],
                    "mAP50 all": round(other.box.map50, 3),
                    "clusters found (ignoring young/ready)": round(one.box.map50, 3),
                    "check (other farms) mAP50 all": round(own.box.map50, 3)})

In [ ]:
# Keep each run's per-epoch history for plotting: one CSV per run in history/, plus the test score, zipped
import datetime, os, re, shutil, yaml
HIST = f"{WORK}/history"
os.makedirs(HIST, exist_ok=True)
TODAY = datetime.date.today()

def save_history(model, run, direction, run_dir, train_set, test_set):
    """Copy run_dir/results.csv (one row per epoch) with the settings in front; returns the file name."""
    df = pd.read_csv(f"{run_dir}/results.csv")
    df.columns = [c.strip() for c in df.columns]
    a = yaml.safe_load(open(f"{run_dir}/args.yaml"))
    front = dict(model=model, run=run, train_set=train_set, test_set=test_set, date=TODAY.isoformat(),
                 epochs=a["epochs"], imgsz=a["imgsz"], batch=a["batch"], base_model=os.path.basename(str(a["model"])))
    for i, (k, v) in enumerate(front.items()):
        df.insert(i, k, v)
    name = "_".join(re.sub(r"[^\w-]+", "", p) for p in (model, run, direction, TODAY.strftime("%Y%m%d"))) + ".csv"
    df.to_csv(f"{HIST}/{name}", index=False)
    return name

def save_scores(model, run, table):
    """The final test score table, next to the histories."""
    t = pd.DataFrame(table)
    for i, (k, v) in enumerate({"model": model, "run": run, "date": TODAY.isoformat()}.items()):
        t.insert(i, k, v)
    t.to_csv(f"{HIST}/{model}_{run}_scores_{TODAY.strftime('%Y%m%d')}.csv", index=False)

for left_out, run in RUNS.items():
    others = "+".join(s for s in PHOTOS if s != left_out)
    print(save_history("harvest_readiness", "run3", f"trainothers_test{left_out}", f"{run}/train", others, left_out))
save_scores("harvest_readiness", "run3", results)
print(shutil.make_archive(f"{WORK}/history", "zip", HIST))   # Output tab: history.zip

## Look at 2 photos from each left-out farm
Dataset's labels (left) vs the model's guess (right).

In [ ]:
# 6. Dataset's labels (left) vs model's guess (right), 2 photos per left-out farm
for left_out, run in RUNS.items():
    best = YOLO(f"{run}/train/weights/best.pt")
    test_imgs = open(f"{run}/test.txt").read().split()
    random.seed(1)
    pick = random.sample(test_imgs, min(2, len(test_imgs)))
    fig, axes = plt.subplots(len(pick), 2, figsize=(14, 5 * len(pick)), squeeze=False)
    for (axl, axr), img in zip(axes, pick):
        im = Image.open(img); W, H = im.size
        axl.imshow(im); axl.axis("off"); axl.set_title("dataset's labels")
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        for line in open(lab).read().splitlines():
            c, x, y, w, h = line.split(); c = int(c)
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            axl.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False,
                                            edgecolor=COLOURS[c], linewidth=2))
        axr.imshow(best.predict(img, imgsz=IMG_SIZE, verbose=False)[0].plot()[:, :, ::-1])
        axr.axis("off"); axr.set_title("model's guess")
    fig.suptitle(f"left out: {left_out}", fontsize=14)
    plt.tight_layout(); plt.show()

In [ ]:
# 7. Keep the models and the score table (Output tab), remove the big photo folders
for left_out, run in RUNS.items():
    shutil.copy(f"{run}/train/weights/best.pt", f"{WORK}/harvest_readiness_leave_out_{left_out}.pt")
for d in glob.glob(f"{WORK}/raw_*") + glob.glob(f"{WORK}/pool_*"):
    shutil.rmtree(d, ignore_errors=True)
summary = pd.DataFrame(results)
summary.to_csv(f"{WORK}/harvest_readiness_scores_3.csv", index=False)
if SKIPPED:
    print("Skipped farms:", SKIPPED)
print("\nSCORE TABLE (screenshot this):")
summary

## Read the result
* **Left-out farm rows.** These are the scores to trust: the model never saw that farm, and its photos were not near-twins of the training photos.
* **clusters found vs young/ready.** Low "clusters found" = the model cannot even find the clusters on a new farm (camera distance, background). High "clusters found" but low young/ready = it finds them but the farms draw the young/ready line differently.
* **Duman row vs the close-up farms.** If only Duman fails, camera distance is the likely cause.
* **Next.** If scores stay low, try photos at similar distances or tiled (cropped) training; syldd photos stay a later, separate test.